# Jais-2-8B-Chat — Query2Doc Query Generator

**Experiment:** exp_006 — Jais-2-8B-Chat + Dense Retrieval  
**Technique:** Query2Doc (pseudo-document generation)  
**Reference baseline:** exp_003 (Qwen 2.5 3B, NDCG@10=0.5435)

## Model Details
- **Model:** `inceptionai/Jais-2-8B-Chat` (Arabic-specialized, 8.09B params)
- **Architecture:** Standard decoder-only Transformer (RoPE, Squared-ReLU, LayerNorm)
- **Developers:** Inception (G42) + MBZUAI + Cerebras (Dec 2024)
- **Training:** 2.6 trillion tokens (Arabic + English + code), SFT + DPO + GRPO
- **Vocab:** 150,272 tokens (Arabic-centric)
- **Context:** 8,192 tokens

## GPU Strategy: A100 (40 GB) — Colab Pro
- **Option A (preferred):** FP16 (~18-20 GB) + batch_size=4-8 → preserves full quality
- **Option B (fallback):** 4-bit NF4 (~6-8 GB) + batch_size=8-16 → if FP16 OOMs
- **Estimated time:** ~30-45 min for 2,896 queries (with batching)

## Jais-2-Specific Notes
- Standard Transformer: NO batching bugs (unlike Falcon-H1's Mamba architecture)
- `apply_chat_template()` works with standard role format
- MUST remove `token_type_ids` from inputs before `generate()` (documented in model card)
- Access gate: must accept terms on HuggingFace before downloading

## Key Research Sources
- Cerebras Blog: "Jais 2: A Blueprint for Sovereign AI" (Dec 2024)
- Original Jais paper: Sengupta et al. (2023), arXiv:2308.16149
- Full research: `research_decisions/jais_2_research.md`

---

## Step 1: Install Dependencies

> After this cell: Runtime → Restart runtime, then continue from Step 2.

In [1]:
# ── Step 1: Install all dependencies ──────────────────────────────────────────
#
# Runtime: A100 (40 GB) — select in Runtime → Change runtime type
#
# Java is required by pyserini (for MIRACL data loading).
# bitsandbytes required for 4-bit quantization (fallback if FP16 OOMs).
#
# After this cell: Runtime → Restart runtime, then continue from Step 2.
# ──────────────────────────────────────────────────────────────────────────────

# 1. Java (required by pyserini — hidden dependency in MIRACLDataLoader)
!apt-get install -qq openjdk-21-jdk-headless

# 2. Retrieval / data loading libraries
!pip install -q pyserini faiss-cpu

# 3. Transformers (Jais-2 architecture requires recent transformers)
# The `jais2` model type was added in transformers v4.47+ (Dec 2025).
# Standard pip install should work; if loading fails, uncomment the next line:
!pip install -q transformers
# !pip install -q git+https://github.com/huggingface/transformers.git

# 4. 4-bit quantization dependencies (for fallback if FP16 OOMs)
!pip install -q bitsandbytes accelerate

# 5. ML / utility libraries
!pip install -q torch datasets tqdm

print("\n" + "=" * 60)
print("Installation complete")
print("=" * 60)
print("IMPORTANT: Restart runtime now!")
print("   Runtime -> Restart runtime")
print("   Then run cells starting from Step 2")
print("=" * 60)

Selecting previously unselected package openjdk-21-jre-headless:amd64.
(Reading database ... 121852 files and directories currently installed.)
Preparing to unpack .../openjdk-21-jre-headless_21.0.10+7-1~22.04_amd64.deb ...
Unpacking openjdk-21-jre-headless:amd64 (21.0.10+7-1~22.04) ...
Selecting previously unselected package openjdk-21-jdk-headless:amd64.
Preparing to unpack .../openjdk-21-jdk-headless_21.0.10+7-1~22.04_amd64.deb ...
Unpacking openjdk-21-jdk-headless:amd64 (21.0.10+7-1~22.04) ...
Setting up openjdk-21-jre-headless:amd64 (21.0.10+7-1~22.04) ...
update-alternatives: using /usr/lib/jvm/java-21-openjdk-amd64/bin/java to provide /usr/bin/java (java) in auto mode
update-alternatives: using /usr/lib/jvm/java-21-openjdk-amd64/bin/jpackage to provide /usr/bin/jpackage (jpackage) in auto mode
update-alternatives: using /usr/lib/jvm/java-21-openjdk-amd64/bin/keytool to provide /usr/bin/keytool (keytool) in auto mode
update-alternatives: using /usr/lib/jvm/java-21-openjdk-amd64/b

## Step 2: Mount Drive and Setup Environment

> Run this after restarting runtime

In [1]:
from google.colab import drive
drive.mount('/content/drive')

# Clone project repo (or pull if already cloned)
!git clone https://github.com/Osmanoor/graduation.git 2>/dev/null || (cd /content/graduation && git pull)
%cd /content/graduation/arabic-rag-query-enhancement

import os
import sys

# Java home required by pyserini
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-21-openjdk-amd64'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

sys.path.insert(0, '/content/graduation/arabic-rag-query-enhancement')

# Verify Java
!java -version

import torch
print(f"\nEnvironment configured")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

import transformers
print(f"Transformers version: {transformers.__version__}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Already up to date.
/content/graduation/arabic-rag-query-enhancement
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.10" 2026-01-20
OpenJDK Runtime Environment (build 21.0.10+7-Ubuntu-122.04)
OpenJDK 64-Bit Server VM (build 21.0.10+7-Ubuntu-122.04, mixed mode, sharing)

Environment configured
GPU Available: True
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB
Transformers version: 5.0.0


## Step 3: Load MIRACL Arabic Data

In [2]:
from src.utils.data_loader import MIRACLDataLoader

data_loader = MIRACLDataLoader(language="ar", split="dev")
topics, qrels = data_loader.load_all()

query_ids = list(topics.keys())
query_texts = [topics[qid]['title'] for qid in query_ids]

print(f"\nDataset Statistics:")
print(f"  Queries: {len(query_ids)}")
print(f"  Qrels:   {len(qrels)}")
print(f"\nSample query: {query_texts[0]}")

Loading topics from miracl-v1.0-ar-dev...
✓ Loaded 2896 queries
Loading qrels from miracl-v1.0-ar-dev...
✓ Loaded qrels for 2896 queries

Dataset Statistics:
  Queries: 2896
  Qrels:   2896

Sample query: من هو علي بن محمد السمري؟


## Step 4: Initialize Jais-2-8B-Chat

**Strategy:** Try FP16 first (preserves full quality). Fall back to 4-bit NF4 if OOM.

**A100 40GB VRAM budget:**
- FP16: ~18-20 GB model → ~20 GB free for batching → batch_size=4-8
- 4-bit: ~6-8 GB model → ~32 GB free for batching → batch_size=8-16

**Architecture:** Standard Transformer (RoPE, Squared-ReLU, MHA with 26 heads)  
No SSM buffers, no batching bugs. Safe to use batch generation.

> **Pre-requisite:** Accept the access gate on  
> https://huggingface.co/inceptionai/Jais-2-8B-Chat  
> and log in via `notebook_login()`

In [ ]:
from huggingface_hub import login

# Log in using your HuggingFace token
# Get your token from: https://huggingface.co/settings/tokens
NEW_TOKEN = "YOUR_HF_TOKEN_HERE"  # <-- Replace with your token (do NOT commit real tokens)
login(token=NEW_TOKEN)

print("\nCredentials updated! You can now try running the model loading cell again.")

In [4]:
# Login to HuggingFace (required for gated model)
from huggingface_hub import notebook_login
notebook_login()

In [1]:
from huggingface_hub import whoami, get_token
import os

print("=== Hugging Face Token Diagnostic ===")
token = get_token()
if token is None:
    print("❌ No Hugging Face token found! `notebook_login()` may have failed or the token wasn't saved.")
    print("   Try running this instead: !huggingface-cli login --token YOUR_TOKEN")
else:
    print("✅ Token found in environment/cache.")
    try:
        user_info = whoami()
        print(f"✅ Logged in as: {user_info.get('name', 'Unknown')}")
        print(f"✅ Token type/role: {user_info.get('auth', {}).get('accessToken', {}).get('role', 'Unknown')}")
        print("\nIf you are logged in but still getting a 403 error:")
        print("1. Ensure THIS exact account accepted the terms at https://huggingface.co/inceptionai/Jais-2-8B-Chat")
        print("2. If using a Fine-Grained token, ensure it has 'Read' access to gated repositories.")
        print("3. Try generating a new standard 'Read' token at https://huggingface.co/settings/tokens")
    except Exception as e:
        print(f"❌ Token is invalid or expired. Error: {e}")

=== Hugging Face Token Diagnostic ===
✅ Token found in environment/cache.
✅ Logged in as: osmanandmohammed
✅ Token type/role: read

If you are logged in but still getting a 403 error:
1. Ensure THIS exact account accepted the terms at https://huggingface.co/inceptionai/Jais-2-8B-Chat
2. If using a Fine-Grained token, ensure it has 'Read' access to gated repositories.
3. Try generating a new standard 'Read' token at https://huggingface.co/settings/tokens


In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "inceptionai/Jais-2-8B-Chat"

# ── Configuration ─────────────────────────────────────────────────────────────
# Strategy: BF16 first (best quality, matches model's native dtype).
# Jais-2 uses Squared-ReLU activation — FP16 overflows (causes CUDA assert).
# BF16 is natively supported on A100 (compute capability 8.0).
# Fall back to 4-bit NF4 only if OOM.
USE_4BIT = False  # Set True to force 4-bit, or leave False to use BF16
BATCH_SIZE = 8    # Start with 8; increase to 16 if VRAM allows
# ──────────────────────────────────────────────────────────────────────────────

print(f"Loading {MODEL_NAME}...")
print(f"Mode: {'4-bit NF4' if USE_4BIT else 'BF16 (model native dtype)'}")
print(f"Target batch size: {BATCH_SIZE}")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'  # Required for decoder-only batch generation

# Load model
if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,  # BF16 compute even in 4-bit
        bnb_4bit_use_double_quant=True
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto"
    )
else:
    # BF16 — matches model's native tensor type.
    # DO NOT use float16: Jais-2's Squared-ReLU activation produces values
    # that overflow FP16 range (max ~65504), causing CUDA device-side assert
    # in torch.multinomial during sampling. BF16 handles up to ~3.4e38.
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )

model.eval()

# Report VRAM usage
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    used = (total - free) / 1e9
    print(f"\nGPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {used:.1f} GB used / {total/1e9:.1f} GB total")
    print(f"Free: {free/1e9:.1f} GB")
    # Suggest batch size based on free VRAM
    if free / 1e9 > 25:
        suggested_bs = 16
    elif free / 1e9 > 15:
        suggested_bs = 8
    elif free / 1e9 > 8:
        suggested_bs = 4
    else:
        suggested_bs = 1
    print(f"Suggested batch size: {suggested_bs} (based on {free/1e9:.1f} GB free)")
    if suggested_bs != BATCH_SIZE:
        print(f"  -> Consider changing BATCH_SIZE to {suggested_bs}")

print(f"\nJais-2-8B-Chat ready ({'4-bit NF4' if USE_4BIT else 'BF16'})")
print(f"  Vocab size: {len(tokenizer)}")
print(f"  Pad token: {tokenizer.pad_token}")

import transformers
print(f"  Transformers: {transformers.__version__}")

Loading inceptionai/Jais-2-8B-Chat...
Mode: BF16 (model native dtype)
Target batch size: 8


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]


GPU: NVIDIA A100-SXM4-40GB
VRAM: 16.6 GB used / 42.4 GB total
Free: 25.8 GB
Suggested batch size: 16 (based on 25.8 GB free)
  -> Consider changing BATCH_SIZE to 16

Jais-2-8B-Chat ready (BF16)
  Vocab size: 150222
  Pad token: <|endoftext|>
  Transformers: 5.0.0


## Step 5: Sanity Check — First 5 Queries

**Check before proceeding to full run:**
- Output is in Arabic (not English, not garbage/repetition)
- Pseudo-document is relevant to the query topic
- Expansion ratio is reasonable (5-12x)
- No error messages or warnings

**Jais-2-specific:** Remove `token_type_ids` from inputs before `generate()`  
(Documented in model card: https://huggingface.co/inceptionai/Jais-2-8B-Chat)

In [6]:
SYSTEM_PROMPT = (
    "You are asked to write a passage that answers the given query. "
    "Do not ask the user for further clarification. "
    "Respond in Arabic only."
)

TEMPERATURE = 0.7
MAX_NEW_TOKENS = 128
TOP_P = 0.9

print("Sanity check: testing on first 5 queries (single-query mode)\n")
print("=" * 60)

for i in range(5):
    query = query_texts[i]

    # Format using chat template (standard HF role format — works for Jais-2)
    chat_text = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": query}
        ],
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(chat_text, return_tensors="pt").to(model.device)
    inputs.pop("token_type_ids", None)  # Jais-2 requirement (model card)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id  # explicitly use eos_token_id
        )

    # Decode only the generated tokens (skip the prompt)
    generated = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    ).strip()

    # Construct enhanced query (Query2Doc format: original + pseudo-doc)
    enhanced = f"{query} {generated}"
    ratio = len(enhanced) / max(len(query), 1)

    print(f"\nQuery {i+1} [{query_ids[i]}]: {query}")
    print(f"Generated ({len(generated)} chars):")
    print(f"{generated[:300]}..." if len(generated) > 300 else generated)
    print(f"Expansion ratio: {ratio:.1f}x")

print("\n" + "=" * 60)
print("Sanity check complete")
print("\nBefore proceeding, verify:")
print("  [ ] Output is in Arabic")
print("  [ ] Content is relevant to query")
print("  [ ] Expansion ratio 5-12x")
print("  [ ] No errors or garbage output")

Sanity check: testing on first 5 queries (single-query mode)


Query 1 [8099]: من هو علي بن محمد السمري؟
Generated (505 chars):
علي بن محمد السمري هو شخصية شيعية بارزة، يُعرف بكونه **آخر نواب الإمام المهدي (عليه السلام)**، أي آخر من ناب عن الإمام المهدي (عليه السلام) في فترة الغيبة الصغرى. وقد لعب دورًا مهمًا في نقل الرسائل والتعاليم من الإمام إلى شيعته خلال هذه الفترة.

**أهم النقاط حول علي بن محمد السمري:**

*   **الغيبة ا...
Expansion ratio: 21.2x

Query 2 [3640]: متى تم إستخدام الغوّاصات لأول مرة؟
Generated (330 chars):
تم استخدام الغواصات لأول مرة في أوائل القرن السابع عشر. كانت أولى الغواصات الحديثة المعروفة قد ظهرت في أواخر القرن السادس عشر، ولكن الاستخدام العسكري الفعلي لها بدأ في حوالي عام 1620. كانت هذه الغواصات المبكرة، مثل "البرنيق" (البرنقيل)، عبارة عن هياكل بدائية تعمل بدفع من المجداف وتستخدم في الغالب لأ...
Expansion ratio: 10.7x

Query 3 [4971]: من هو القديس المسمى بالصخرة؟
Generated (41 chars):
القديس المسمى بالصخرة هو **القديس بطرس**.
Expansion ratio: 2.5x

Query 4 [1

## Step 6: Full Generation — 2,896 Queries (Batched)

**Mode:** Batched generation (BATCH_SIZE from Step 4)  
**Why batching works:** Jais-2 is a standard Transformer — no Mamba SSM batching bugs  
**Expected time:** ~30-45 min on A100 with batch_size=8  
**Checkpoints:** Saves progress every 200 queries to pkl  

**Lesson from Falcon-H1 (exp_005):** Falcon's hybrid Mamba architecture forced  
single-query mode. Jais-2 doesn't have this limitation — use batching to maximize  
your Colab Pro subscription.

In [7]:
BATCH_SIZE = 16

In [8]:
import time
import pickle
from tqdm.notebook import tqdm

CHECKPOINT_EVERY = 200
CHECKPOINT_PATH = 'enhanced_queries_jais_2_8b_chat_checkpoint.pkl'

def generate_batch(batch_queries):
    """Generate pseudo-documents for a batch of queries.

    Uses left-padded tokenization for parallel generation.
    Removes token_type_ids (Jais-2 requirement).
    """
    # Format all queries using chat template
    batch_texts = []
    for query in batch_queries:
        chat_text = tokenizer.apply_chat_template(
            [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": query}
            ],
            tokenize=False,
            add_generation_prompt=True
        )
        batch_texts.append(chat_text)

    # Tokenize with left-padding for batch generation
    inputs = tokenizer(
        batch_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    ).to(model.device)
    inputs.pop("token_type_ids", None)  # Jais-2 requirement

    input_length = inputs['input_ids'].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id
        )

    # Decode only the generated tokens (skip prompt)
    generated_texts = tokenizer.batch_decode(
        outputs[:, input_length:],
        skip_special_tokens=True
    )

    # Combine: original query + pseudo-document
    enhanced = [
        f"{q} {g.strip()}" for q, g in zip(batch_queries, generated_texts)
    ]
    return enhanced


print("=" * 60)
quant_mode = '4-bit NF4' if USE_4BIT else 'FP16'
print(f"FULL RUN: Jais-2-8B-Chat ({quant_mode}, temp={TEMPERATURE})")
print(f"Queries: {len(query_texts)}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Mode: Batched (batch_size={BATCH_SIZE})")
print(f"Checkpoints: every {CHECKPOINT_EVERY} queries")
num_batches = (len(query_texts) + BATCH_SIZE - 1) // BATCH_SIZE
est_time = num_batches * 3 / 60  # ~3 sec per batch rough estimate
print(f"Estimated: {num_batches} batches, ~{est_time:.0f} min")
print("=" * 60 + "\n")

start_time = time.time()
enhanced_queries = []
errors = []

# Process in batches
for batch_idx in tqdm(range(num_batches), desc="Enhancing queries"):
    start_idx = batch_idx * BATCH_SIZE
    end_idx = min(start_idx + BATCH_SIZE, len(query_texts))
    batch_queries = query_texts[start_idx:end_idx]
    batch_qids = query_ids[start_idx:end_idx]

    try:
        batch_enhanced = generate_batch(batch_queries)
        enhanced_queries.extend(batch_enhanced)
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            # OOM: fall back to single-query for this batch
            print(f"\nOOM at batch {batch_idx}! Falling back to single-query mode...")
            torch.cuda.empty_cache()
            for j, (q, qid) in enumerate(zip(batch_queries, batch_qids)):
                try:
                    result = generate_batch([q])
                    enhanced_queries.extend(result)
                except Exception as e2:
                    print(f"  Error on query {start_idx+j} [{qid}]: {e2}")
                    errors.append((start_idx+j, qid, str(e2)))
                    enhanced_queries.append(q)  # Fallback to original
        else:
            # Non-OOM error: log and fall back for entire batch
            print(f"\nError at batch {batch_idx}: {e}")
            for j, (q, qid) in enumerate(zip(batch_queries, batch_qids)):
                errors.append((start_idx+j, qid, str(e)))
                enhanced_queries.append(q)

    # Checkpoint
    completed = len(enhanced_queries)
    if completed % CHECKPOINT_EVERY < BATCH_SIZE and completed > 0:
        with open(CHECKPOINT_PATH, 'wb') as f:
            pickle.dump({
                'enhanced_so_far': enhanced_queries,
                'completed': completed,
                'total': len(query_texts)
            }, f)

elapsed = time.time() - start_time
print(f"\nEnhanced {len(enhanced_queries)} queries in {elapsed/60:.1f} minutes")
print(f"  Speed: {len(enhanced_queries) / (elapsed/60):.1f} queries/minute")
print(f"  Batch size: {BATCH_SIZE}")
if errors:
    print(f"  Errors: {len(errors)} (fell back to original query)")
    for idx, qid, err in errors[:5]:
        print(f"    Query {idx} [{qid}]: {err}")

FULL RUN: Jais-2-8B-Chat (FP16, temp=0.7)
Queries: 2896
GPU: NVIDIA A100-SXM4-40GB
Mode: Batched (batch_size=16)
Checkpoints: every 200 queries
Estimated: 181 batches, ~9 min



Enhancing queries:   0%|          | 0/181 [00:00<?, ?it/s]


Enhanced 2896 queries in 12.0 minutes
  Speed: 241.5 queries/minute
  Batch size: 16


## Step 7: Save Results

In [9]:
import pickle
from datetime import datetime

quant_mode = '4-bit NF4 (bitsandbytes, double-quant)' if USE_4BIT else 'BF16 (model native dtype)'

data = {
    'query_ids': query_ids,
    'original': query_texts,
    'enhanced': enhanced_queries,
    'metadata': {
        'model': MODEL_NAME,
        'architecture': 'Standard Transformer (RoPE, Squared-ReLU, LayerNorm)',
        'developers': 'Inception (G42) + MBZUAI + Cerebras',
        'training_tokens': '2.6 trillion',
        'quantization': quant_mode,
        'temperature': TEMPERATURE,
        'max_new_tokens': MAX_NEW_TOKENS,
        'top_p': TOP_P,
        'batch_size': BATCH_SIZE,
        'gpu': torch.cuda.get_device_name(0),
        'technique': 'query2doc',
        'dataset': 'miracl-ar-dev',
        'date': datetime.now().isoformat(),
        'num_queries': len(query_ids),
        'runtime_minutes': round(elapsed / 60, 1),
        'queries_per_minute': round(len(enhanced_queries) / (elapsed / 60), 1),
        'errors': len(errors),
        'system_prompt': SYSTEM_PROMPT,
        'research_doc': 'research_decisions/jais_2_research.md'
    }
}

# Save locally in Colab
local_path = 'enhanced_queries_jais_2_8b_chat.pkl'
with open(local_path, 'wb') as f:
    pickle.dump(data, f)
print(f"Saved locally: {local_path}")

# Save to Google Drive for persistence
drive_base = '/content/drive/MyDrive/graduation project/colab_data'
os.makedirs(drive_base, exist_ok=True)
drive_path = f'{drive_base}/enhanced_queries_jais_2_8b_chat.pkl'
with open(drive_path, 'wb') as f:
    pickle.dump(data, f)
print(f"Saved to Drive: {drive_path}")

print(f"\nKey metadata:")
print(f"  Model: {data['metadata']['model']}")
print(f"  Quantization: {data['metadata']['quantization']}")
print(f"  Batch size: {data['metadata']['batch_size']}")
print(f"  Runtime: {data['metadata']['runtime_minutes']} min")
print(f"  Speed: {data['metadata']['queries_per_minute']} queries/min")

Saved locally: enhanced_queries_jais_2_8b_chat.pkl
Saved to Drive: /content/drive/MyDrive/graduation project/colab_data/enhanced_queries_jais_2_8b_chat.pkl

Key metadata:
  Model: inceptionai/Jais-2-8B-Chat
  Quantization: BF16 (model native dtype)
  Batch size: 16
  Runtime: 12.0 min
  Speed: 241.5 queries/min


## Step 8: Expansion Statistics

In [10]:
import numpy as np

print("=" * 60)
print("EXPANSION STATISTICS")
print("=" * 60)

orig_lens = [len(q) for q in query_texts]
enh_lens = [len(q) for q in enhanced_queries]
ratios = [e / max(o, 1) for e, o in zip(enh_lens, orig_lens)]

print(f"\nJais-2-8B-Chat ({'4-bit' if USE_4BIT else 'FP16'}, temp={TEMPERATURE}):")
print(f"  Avg original length : {np.mean(orig_lens):.1f} chars")
print(f"  Avg enhanced length : {np.mean(enh_lens):.1f} chars")
print(f"  Avg expansion ratio : {np.mean(ratios):.2f}x")
print(f"  Median expansion    : {np.median(ratios):.2f}x")
print(f"  Min expansion       : {np.min(ratios):.2f}x")
print(f"  Max expansion       : {np.max(ratios):.2f}x")

print("\n" + "=" * 60)
print("REFERENCE (previous experiments):")
print("  exp_003 Qwen 2.5 3B:     Avg 9.73x (247.6 chars)")
print("  exp_005 Falcon-H1-3B:    See exp_005 doc")
print("=" * 60)

print(f"\nPkl file saved. Next step:")
print(f"  Open evaluate_enhanced_queries.ipynb")
print(f"  Upload {local_path} for Dense retrieval evaluation")

EXPANSION STATISTICS

Jais-2-8B-Chat (FP16, temp=0.7):
  Avg original length : 29.5 chars
  Avg enhanced length : 256.0 chars
  Avg expansion ratio : 10.46x
  Median expansion    : 5.04x
  Min expansion       : 1.13x
  Max expansion       : 51.00x

REFERENCE (previous experiments):
  exp_003 Qwen 2.5 3B:     Avg 9.73x (247.6 chars)
  exp_005 Falcon-H1-3B:    See exp_005 doc

Pkl file saved. Next step:
  Open evaluate_enhanced_queries.ipynb
  Upload enhanced_queries_jais_2_8b_chat.pkl for Dense retrieval evaluation


---

## Retrieval Evaluation Results

### Dense Retrieval (mDPR)

| Metric | Baseline (mDPR) | Jais-2-8B | Change |
|--------|-----------------|-----------|--------|
| **Recall@10** | 0.6156 | **0.7161** | **+16.3%** |
| **Recall@100** | 0.8407 | **0.8981** | **+6.8%** |
| **NDCG@10** | 0.4993 | **0.6018** | **+20.5%** |
| **MRR** | 0.5328 | **0.6356** | **+19.3%** |

### BM25 Retrieval (BM25S)

| Metric | Baseline (BM25) | Jais-2-8B | Change |
|--------|-----------------|-----------|--------|
| **Recall@10** | 0.5964 | **0.6448** | **+8.1%** |
| **Recall@100** | 0.8577 | **0.8834** | **+3.0%** |
| **NDCG@10** | 0.4621 | **0.5122** | **+10.8%** |
| **MRR** | 0.4836 | **0.5397** | **+11.6%** |

### Full Comparison (Dense)

| Model | NDCG@10 | Recall@10 | Recall@100 | MRR |
|-------|---------|-----------|------------|-----|
| mDPR baseline | 0.4993 | 0.6156 | 0.8407 | 0.5328 |
| Qwen 2.5 3B (exp_003) | 0.5435 | 0.6608 | 0.8594 | 0.5742 |
| Falcon-H1-3B (exp_005) | 0.5359 | 0.6484 | 0.8531 | 0.5681 |
| **Jais-2-8B (exp_006)** | **0.6018** | **0.7161** | **0.8981** | **0.6356** |

---

## Lessons Learned

### Technical
1. **FP16 does NOT work** — Jais-2's Squared-ReLU activation overflows FP16 range (max 65,504), causing `CUDA device-side assert` in `torch.multinomial`. Must use BF16 on A100 (native support, compute capability 8.0). Same issue as Falcon-H1.
2. **BF16 on A100:** 16.6 GB VRAM used (no quantization needed). 25.8 GB free for batching.
3. **Batch size 16** worked perfectly — no OOM, 0 errors on 2,896 queries.
4. **Runtime: 12.0 min** (241.5 queries/min) — 3x faster than Qwen 2.5 3B (~40 min) and 5x faster than Falcon-H1 (~60-90 min). Batching on standard Transformer + A100 = massive speedup.
5. **`token_type_ids` removal** worked as documented — no other quirks encountered.
6. **`torch_dtype` deprecated** in transformers 5.0.0 — use `dtype` instead (cosmetic warning only).

### Research
1. **Best model by a wide margin:** +20.5% NDCG@10 over baseline — more than double the improvement of Qwen (+8.9%) and Falcon-H1 (+7.3%).
2. **First model to improve BM25:** Qwen 2.5 3B hurt BM25 (-11.5% NDCG). Jais-2 improved it by +10.8%. This suggests Arabic-specialized vocabulary produces more lexically relevant expansion terms.
3. **Concise expansions work better:** Despite lower median expansion (5.04x vs Qwen's 8.45x), Jais-2 achieved much better retrieval. Quality > quantity for expansion terms.
4. **Arabic specialization matters:** 150K Arabic-centric vocab + 2.6T tokens with heavy Arabic representation = better expansion quality than multilingual models.

### Key Finding
**Jais-2-8B-Chat is the clear leader for Arabic query expansion.** It is the fastest to run (12 min), easiest to implement (standard Transformer), and produces the best retrieval results across both Dense (+20.5%) and BM25 (+10.8%). The combination of Arabic-specialized training + larger parameter count (8B vs 3B) produces significantly better expansions than multilingual alternatives.

### Generation Phase Comparison

| Model | Params | GPU | Precision | Batch | Runtime | Speed | Errors |
|-------|--------|-----|-----------|-------|---------|-------|--------|
| Qwen 2.5 3B | 3B | T4 | FP16 | 8 | ~40 min | ~72 q/min | 0 |
| Falcon-H1-3B | 3.15B | A100 | BF16 | 1 | ~60-90 min | ~32-48 q/min | 0 |
| **Jais-2-8B** | **8.09B** | **A100** | **BF16** | **16** | **12 min** | **241.5 q/min** | **0** |

---

## Citations

- Wang, L., Yang, N., & Wei, F. (2023). Query2doc: Query Expansion with Large Language Models. arXiv:2303.07678.
- Sengupta, N., et al. (2023). Jais and Jais-chat: Arabic-Centric Foundation and Instruction-Tuned Open Generative Large Language Models. arXiv:2308.16149.
- Cerebras (2024). Jais 2: A Blueprint for Sovereign AI. https://www.cerebras.ai/blog/jais2
- Zhang, X., et al. (2023). MIRACL: A Multilingual Retrieval Dataset. TACL.
- Research notes: `research_decisions/jais_2_research.md`